MANC notebook- create csv annotation fiels for MANC in version 1.2.3. 2026 04 03

In [19]:
# set up environment
import sys
import json
import re
import requests
import pandas as pd
from pathlib import Path
import json

from caveclient import CAVEclient

import urllib.parse


print("Python executable:", sys.executable)
print("Imports OK")

Python executable: c:\Users\jsfdz\Documents\Python\cave_env\Scripts\python.exe
Imports OK


In [16]:
from pathlib import Path
import json

# set up directories
PROJECT_ROOT = Path.cwd()

INPUT_DIR = PROJECT_ROOT / "output"
if not INPUT_DIR.exists():
    raise FileNotFoundError(f"Input directory missing: {INPUT_DIR}")

OUTPUT_DIR = PROJECT_ROOT / "output-MANCv1.2.3_04B_morphology_clusters_T1"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Input:", INPUT_DIR)
print("Output:", OUTPUT_DIR)

# load input state
STATE_IN = INPUT_DIR / "manc_v1p2_with_manual_IR_layers.json"
if not STATE_IN.exists():
    raise FileNotFoundError(f"Missing state file: {STATE_IN}")

with open(STATE_IN, "r", encoding="utf-8") as f:
    state = json.load(f)

# save output state
STATE_OUT = OUTPUT_DIR / "manc_4b_morphology_clusters_jhs.json"

with open(STATE_OUT, "w", encoding="utf-8") as f:
    json.dump(state, f, indent=2)

Input: c:\Users\jsfdz\4B-analysis\4B-analysis\output
Output: c:\Users\jsfdz\4B-analysis\4B-analysis\output-MANCv1.2.3_04B_morphology_clusters_T1


In [14]:
# build a simple descriptive CSV to describe the state view
import json
import csv
import re

# ---- LOAD INPUT STATE (read only) ----
with open(STATE_IN, "r", encoding="utf-8") as f:
    state = json.load(f)

# ---- PREP OUTPUT ----
state_title = state.get("title", "untitled_state")
safe_title = re.sub(r"[^a-zA-Z0-9_-]+", "_", state_title)

OUT_CSV = OUTPUT_DIR / f"{safe_title}_layer_annotations.csv"

# ---- EXTRACT LAYERS ----
rows = []

for L in state.get("layers", []):
    layer_type = L.get("type", "")
    name = L.get("name", "")
    visible = L.get("visible", "")

    segments = L.get("segments", [])
    segments_IDs = ",".join(str(s) for s in segments) if segments else ""

    seg_colors = L.get("segmentColors") or {}
    unique_colors = set(seg_colors.values())
    layer_color = list(unique_colors)[0] if len(unique_colors) == 1 else ""

    rows.append({
        "state_title": state_title,
        "layer_name": name,
        "layer_type": layer_type,
        "visible": visible,
        "layer_color": layer_color,
        "num_segments": len(segments),
        "segments_IDs": segments_IDs,
        "note": ""
    })

# ---- WRITE CSV ----
fieldnames = [
    "state_title",
    "layer_name",
    "layer_type",
    "visible",
    "layer_color",
    "num_segments",
    "segments_IDs",
    "note",
]

# define output filename based on input state
OUT_CSV = OUTPUT_DIR / f"{STATE_IN.stem}_layer_summary.csv"

# write CSV
with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"Wrote summary of {STATE_IN.name} → {OUT_CSV.name}")

Wrote summary of manc_v1p2_with_manual_IR_layers.json → manc_v1p2_with_manual_IR_layers_layer_summary.csv


Make a csv with neuron as row, and grouped by sefment alyer, and add info liek segment ID and MANC type 

In [15]:
# build a CSV mapping segment IDs to manual morphology clusters
import json
import csv

# ---- LOAD INPUT STATE ----
with open(STATE_IN, "r", encoding="utf-8") as f:
    state = json.load(f)

# ---- DEFINE OUTPUT ----
OUT_CSV = OUTPUT_DIR / f"{STATE_IN.stem}_manual_clusters_by_segment.csv"

rows = []

state_title = state.get("title", "untitled_state")

# ---- EXTRACT PER-SEGMENT DATA ----
for L in state.get("layers", []):
    layer_name = L.get("name", "")
    layer_type = L.get("type", "")
    visible = L.get("visible", "")

    segments = L.get("segments", [])

    for seg_id in segments:
        rows.append({
            "state_title": state_title,
            "segment_id": str(seg_id),
            "manual_morphology_cluster": layer_name,
            "manc_type": "",  # to be filled later
            "layer_type": layer_type,
            "visible": visible,
            "note": ""
        })

# ---- WRITE CSV ----
fieldnames = [
    "state_title",
    "segment_id",
    "manual_morphology_cluster",
    "manc_type",
    "layer_type",
    "visible",
    "note",
]

with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"Wrote segment → cluster mapping: {OUT_CSV.name}")

Wrote segment → cluster mapping: manc_v1p2_with_manual_IR_layers_manual_clusters_by_segment.csv


Lets make a table for the layers of interest only

In [ ]:

# Build one master DataFrame with one row per segment ID
import json
import pandas as pd

# ---- LOAD INPUT STATE ----
with open(STATE_IN, "r", encoding="utf-8") as f:
    state = json.load(f)

rows = []

for L in state.get("layers", []):
    layer_name = L.get("name", "")
    layer_type = L.get("type", "")
    visible = L.get("visible", "")
    segment_ids = L.get("segments", []) or []

    if layer_type != "segmentation":
        continue
    if not segment_ids:
        continue

    for seg_id in segment_ids:
        rows.append({
            "manual_morphology_cluster": layer_name,
            "layer_type": layer_type,
            "visible": visible,
            "segment_id": int(seg_id),
            "manc_type": "",   # fill later
            "note": ""
        })

all_layers_df = (
    pd.DataFrame(rows)
    .drop_duplicates()
    .sort_values(["manual_morphology_cluster", "segment_id"])
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

all_layers_df

,manual_morphology_cluster,layer_type,visible,segment_id,manc_type,note
0,all-synaptic,segmentation,,1,,
1,all-tissue,segmentation,False,1,,
2,court et al. tracts,segmentation,,100,,
3,court et al. tracts,segmentation,,101,,
4,court et al. tracts,segmentation,,102,,
5,court et al. tracts,segmentation,,103,,
6,court et al. tracts,segmentation,,104,,
7,court et al. tracts,segmentation,,105,,
8,court et al. tracts,segmentation,,106,,
9,manc-seg-v1.2 4B manual IR- proprioceptice ta...,segmentation,False,21862,,


In [ ]:
#inspect layer/ morphology cluster names
for name in sorted(all_layers_df["manual_morphology_cluster"].unique()):
    print(name)

all-synaptic
all-tissue
court et al. tracts
manc-seg-v1.2 4B manual  IR- proprioceptice tactile n=3
manc-seg-v1.2 4B manual  IR-nomorphology assigned-early secondary
manc-seg-v1.2 4B manual BR-1 n=6
manc-seg-v1.2 4B manual BR-2 n=10
manc-seg-v1.2 4B manual BR-3 n=4
manc-seg-v1.2 4B manual IR nomorphology assigned secondary
manc-seg-v1.2 4B manual IR-1
manc-seg-v1.2 4B manual IR-2
manc-seg-v1.2 4B manual IR-3
manc-seg-v1.2 4B manual IR-4
manc-seg-v1.2 4B manual IR-independent leg n=11
manc-seg-v1.2 4B- primary  n=4
manc-seg-v1.2 4b-manual BR-4 n=6
manc-seg-v1.2-4B-all
manc-seg-v1.2-4B-subbclass BI n=4
manc-seg-v1.2-4B-sublass IR n=60
manc-seg-v1.21- secondary n=73
manc-seg-v1.21-4B subclass BA n=1
manc-seg-v1.21-4B-subclassBR n=26
manc-seg-v1.22-4B subclass IA=4
manc-seg-v1.22-4B-early secondary n= 20
manc:v1.2.1-TTMn
nerves
neuropils


In [21]:
LAYERS_TO_ANALYZE = [
    "manc-seg-v1.2 4B manual  IR- proprioceptice tactile n=3",
    "manc-seg-v1.2 4B manual BR-1 n=6",
    "manc-seg-v1.2 4B manual BR-2 n=10",
    "manc-seg-v1.2 4B manual BR-3 n=4",
    "manc-seg-v1.2 4B manual IR-1",
    "manc-seg-v1.2 4B manual IR-2",
    "manc-seg-v1.2 4B manual IR-3",
    "manc-seg-v1.2 4B manual IR-4",
    "manc-seg-v1.2 4B manual IR-independent leg n=11",
    "manc-seg-v1.2 4b-manual BR-4 n=6",
    "manc-seg-v1.2-4B-subbclass BI n=4",
    "manc-seg-v1.2-4B-sublass IR n=60",
    "manc-seg-v1.21-4B subclass BA n=1",
    "manc-seg-v1.21-4B-subclassBR n=26",
    "manc-seg-v1.22-4B subclass IA=4",
]

selected_layers_df = (
    all_layers_df[
        all_layers_df["manual_morphology_cluster"].isin(LAYERS_TO_ANALYZE)
    ]
    .copy()
    .sort_values(["manual_morphology_cluster", "segment_id"])
    .reset_index(drop=True)
)

selected_layers_df

,manual_morphology_cluster,layer_type,visible,segment_id,manc_type,note
0,manc-seg-v1.2 4B manual IR- proprioceptice ta...,segmentation,False,21862,,
1,manc-seg-v1.2 4B manual IR- proprioceptice ta...,segmentation,False,152655,,
2,manc-seg-v1.2 4B manual IR- proprioceptice ta...,segmentation,False,166421,,
3,manc-seg-v1.2 4B manual BR-1 n=6,segmentation,False,13455,,
4,manc-seg-v1.2 4B manual BR-1 n=6,segmentation,False,13506,,
5,manc-seg-v1.2 4B manual BR-1 n=6,segmentation,False,13589,,
6,manc-seg-v1.2 4B manual BR-1 n=6,segmentation,False,14728,,
7,manc-seg-v1.2 4B manual BR-1 n=6,segmentation,False,14774,,
8,manc-seg-v1.2 4B manual BR-1 n=6,segmentation,False,15005,,
9,manc-seg-v1.2 4B manual BR-2 n=10,segmentation,False,10024,,


In [23]:
# check if this ID list is compelte and does nto have duplicates

#is the same neuron added to multipel morphology clusters?
dupes = selected_layers_df[
    selected_layers_df.duplicated(subset="segment_id", keep=False)
].sort_values("segment_id")

print("Total duplicated segment IDs:", dupes["segment_id"].nunique())
dupes


#is it complete?

# ---- define the reference layer: 4B all ----
REFERENCE_LAYER = "manc-seg-v1.2-4B-all"   # replace with exact layer name from your state

# ---- get all IDs from the reference layer ----
layer_4b_all_df = (
    all_layers_df[
        all_layers_df["manual_morphology_cluster"] == REFERENCE_LAYER
    ]
    .copy()
    .sort_values("segment_id")
    .reset_index(drop=True)
)

layer_4b_all_ids = set(layer_4b_all_df["segment_id"])

# ---- get all IDs from your selected layers ----
selected_ids = set(selected_layers_df["segment_id"])

# ---- compare coverage ----
missing_from_selection = sorted(layer_4b_all_ids - selected_ids)
extra_in_selection = sorted(selected_ids - layer_4b_all_ids)

print("IDs in 4B all:", len(layer_4b_all_ids))
print("IDs in selected layers:", len(selected_ids))
print("Missing from selection:", len(missing_from_selection))
print("Extra in selection:", len(extra_in_selection))

Total duplicated segment IDs: 10
IDs in 4B all: 97
IDs in selected layers: 76
Missing from selection: 21
Extra in selection: 0


# Fix these errors when I review the updated version of MANC 

In [ ]:
#----Imports
import os
import json
import pandas as pd
from pathlib import Path
#pip install neuprint-python
from neuprint import Client, fetch_neurons, fetch_custom, NeuronCriteria as NC


0.6.1


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()  # loads .env from current working directory

token = os.environ["NEUPRINT_APPLICATION_CREDENTIALS"]

In [ ]:
from neuprint import Client

c = Client(
    "https://neuprint.janelia.org",
    dataset="manc:v1.2.3",
    token=token
)
print(c)

In [7]:
from dotenv import load_dotenv
import os
from neuprint import Client

load_dotenv()
token = os.environ["NEUPRINT_APPLICATION_CREDENTIALS"]

c = Client(
    "https://neuprint.janelia.org",
    dataset="manc:v1.2.3",
    token=token
)

print(c.fetch_version())

1.7.10


In [39]:
#generate new, clean state:

new_state = {
    "title": "MANC_v1.2.3_4B_morphology_clusters_JHS",
    "layout": "3d",
    "showSlices": True,
    "projectionBackgroundColor": "#ffffff",
    "crossSectionBackgroundColor": "#808080",
    "layerListPanel": {"visible": True},
    "layers": [
        {
            "type": "image",
            "name": "EM",
            "source": em_source,
            "visible": True
        }
    ]
}

In [35]:
#populate one neuromere with 04B 

from neuprint import fetch_custom

cypher = """
MATCH (n:Neuron)
WHERE n.hemilineage = '04B'
  AND n.somaNeuromere = 'T1'
  AND n.somaSide = 'LHS'
RETURN
  n.bodyId AS bodyId,
  n.type AS type,
  n.instance AS instance,
  n.class AS class,
  n.subclass AS subclass,
  n.hemilineage AS hemilineage,
  n.somaNeuromere AS somaNeuromere,
  n.somaSide AS somaSide,
  n.rootSide AS rootSide,
  n.longTract AS longTract,
  n.birthtime AS birthtime
ORDER BY bodyId
"""

df_4b_t1_lhs = fetch_custom(cypher, client=c)
#df_4b_t1_lhs.head() 
#df_4b_t1_lhs["bodyId"].nunique() # n=97

In [41]:
# Populate: split neurons by subclass WITH colors

import json
import urllib.parse
from itertools import cycle

# --- Extract sources ---
seg_source = None
em_source = None

for L in state["layers"]:
    layer_type = L.get("type")
    src = L.get("source")

    if layer_type == "segmentation" and seg_source is None:
        url = src.get("url") if isinstance(src, dict) else src
        if "manc-seg-v1p2/manc-seg-v1.2" in str(url):
            seg_source = src

    if layer_type == "image" and em_source is None:
        em_source = src

assert seg_source is not None, "Segmentation source not found"
assert em_source is not None, "EM source not found"



# --- Color palette (cycles if needed) ---
COLORS = [
    "#87CEEB",  # sky blue
    "#FFA500",  # orange
    "#E34234",  # vermillion
    "#CC99FF",  # pale violet
    "#66CDAA",  # medium aquamarine
    "#FFD700",  # gold
]

color_cycle = cycle(COLORS)


# --- Helper with segmentColors (KEY UPGRADE) ---
def add_segmentation_layer(state, name, body_ids, seg_source, color):
    body_ids = sorted(set(int(x) for x in body_ids))
    
    state["layers"].append({
        "type": "segmentation",
        "name": name,
        "source": seg_source,
        "segments": [str(x) for x in body_ids],
        "segmentColors": {str(x): color for x in body_ids},  # <-- critical
        "visible": True,
    })


# --- Clean subclass labels ---
subclass_df = df_4b_t1_lhs.copy()
subclass_df["subclass"] = subclass_df["subclass"].fillna("unclassified").astype(str).str.strip()
subclass_df.loc[subclass_df["subclass"] == "", "subclass"] = "unclassified"


# --- Inspect counts ---
subclass_counts = (
    subclass_df.groupby("subclass")["bodyId"]
    .nunique()
    .sort_values(ascending=False)
)
print(subclass_counts)


# --- Build layers with colors ---
for subclass_name, group in subclass_df.groupby("subclass", sort=True):
    body_ids = group["bodyId"].dropna().astype(int).tolist()
    layer_name = f"MANC_v1.2.3_{subclass_name}"
    
    color = next(color_cycle)
    
    add_segmentation_layer(new_state, layer_name, body_ids, seg_source, color)


# --- Debug print ---
print("State title:", new_state["title"])
print("Number of layers:", len(new_state["layers"]))

for layer in new_state["layers"]:
    print(layer["name"], len(layer.get("segments", [])))


# --- Save ---
OUTPUT_DIR.mkdir(exist_ok=True)

STATE_OUT = OUTPUT_DIR / "MANC_v1.2.3_4B_morphology_clusters_JHS.json"

with open(STATE_OUT, "w", encoding="utf-8") as f:
    json.dump(new_state, f, indent=2)


# --- Generate Neuroglancer URL ---
encoded = urllib.parse.quote(json.dumps(new_state, separators=(",", ":")))
NEW_URL = "https://clio-ng.janelia.org/#!" + encoded

print("Saved new state:", STATE_OUT)
print("\nNew URL:\n")
print(NEW_URL)

subclass
IR    60
BR    26
BI     6
IA     4
BA     1
Name: bodyId, dtype: int64
State title: MANC_v1.2.3_4B_morphology_clusters_JHS
Number of layers: 11
EM 0
MANC_v1.2.3_BA 1
MANC_v1.2.3_BI 6
MANC_v1.2.3_BR 26
MANC_v1.2.3_IA 4
MANC_v1.2.3_IR 60
MANC_v1.2.3_BA 1
MANC_v1.2.3_BI 6
MANC_v1.2.3_BR 26
MANC_v1.2.3_IA 4
MANC_v1.2.3_IR 60
Saved new state: c:\Users\jsfdz\4B-analysis\4B-analysis\output-MANCv1.2.3_04B_morphology_clusters_T1\MANC_v1.2.3_4B_morphology_clusters_JHS.json

New URL:

https://clio-ng.janelia.org/#!%7B%22title%22%3A%22MANC_v1.2.3_4B_morphology_clusters_JHS%22%2C%22layout%22%3A%223d%22%2C%22showSlices%22%3Atrue%2C%22projectionBackgroundColor%22%3A%22%23ffffff%22%2C%22crossSectionBackgroundColor%22%3A%22%23808080%22%2C%22layerListPanel%22%3A%7B%22visible%22%3Atrue%7D%2C%22layers%22%3A%5B%7B%22type%22%3A%22image%22%2C%22name%22%3A%22EM%22%2C%22source%22%3A%7B%22url%22%3A%22precomputed%3A//gs%3A//flyem-vnc-2-26-213dba213ef26e094c16c860ae7f4be0/v3_emdata_clahe_xy/jpeg%22%2C%2